In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict,Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from pydantic import BaseModel,Field


load_dotenv()

In [ ]:
model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
)

class sentimentSchema(BaseModel):
    sentiment:Literal['positive','negative'] = Field(description="either feedback is positive or negative.")

sentiment_checker_model = model.with_structured_output(sentimentSchema)

class evalSchema(BaseModel):
    issue_type :Literal['UX','Performance','Bug','Support','Other'] = Field(description="Type of issue is user facing")
    urgency :Literal['Low','Medium','High'] = Field(description="how urgent is the issue to be sovled.")
    user_tone :Literal['Angry','Frustrated','Disappointed','Calm'] = Field(description="the emotional tone expressed by the user")

eval_model = model.with_structured_output(evalSchema)

In [ ]:
class FeedbackState(TypedDict):
    feedback: str
    sentiment: Literal['positive','negative']
    diagnosis:dict
    response: str


In [ ]:
def find_sentiment(state:FeedbackState):
    ans = sentiment_checker_model.invoke(state["feedback"])
    return {"sentiment":ans.sentiment}
    
def run_diagnosis(state:FeedbackState):
    res = eval_model.invoke(state["feedback"])
    return {"diagnosis": res.model_dump()}
    
def positive_response(state:FeedbackState):
    prompt = f"Based on this feedback, create a warm thank you message for the user. \n feedback: {state['feedback']}"
    ans = model.invoke(prompt)
    return {"response": ans.content}
    
def negative_response(state:FeedbackState):
    diagnosis = state["diagnosis"]
    prompt = f"You are a supportive assistant. The user had {diagnosis['issue_type']} issue, marked urgency as {diagnosis['urgency']} and sounded like {diagnosis['user_tone']}. create an empathetic, helpful resolution message."
    res = model.invoke(prompt).content
    return {"response":res}

def check_condition(state:FeedbackState) -> Literal["positive_response","run_diagnosis"]:
    if state['sentiment'] == "positive":
        return "positive_response"
    else:
        return "run_diagnosis"


In [ ]:
graph = StateGraph(FeedbackState)

graph.add_node("find_sentiment",find_sentiment)
graph.add_node("positive_response",positive_response)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("negative_response",negative_response)

graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges("find_sentiment",check_condition)
graph.add_edge("positive_response",END)
graph.add_edge("run_diagnosis","negative_response")
graph.add_edge("negative_response",END)

workflow = graph.compile()

In [ ]:
initial_state = {"feedback":"The app is too bad. i have ordered my shoes for 6 days, but i have not recieved any. the rider do not respond. they deducted the payment online and now no one is responding. where is my parcel? is it lost? what do i do now? its very frustrating now."}

final_state = workflow.invoke(initial_state)
print(f"final state is: {final_state}")